Python 适配器模式

适配器模式是一种结构型设计模式，它允许不兼容的接口之间进行协作。就像现实生活中的电源适配器可以让不同国家的电器插头连接到本地插座一样，在编程中，适配器模式充当两个不兼容接口之间的桥梁。

核心概念

适配器模式主要解决接口不匹配的问题。当我们需要使用一个现有的类，但其接口不符合我们的需求时，适配器模式就能派上用场。它通过创建一个中间层（适配器）来转换接口，使得原本不兼容的类能够协同工作。

模式结构
适配器模式包含三个主要角色：

目标接口(Target)：客户端期望的接口

适配器(Adapter)：实现目标接口，并包装适配者

适配者(Adaptee)：需要被适配的现有类

为什么需要适配器模式

实际问题场景

想象一下，你有一个现有的日志系统，它使用特定的方法记录日志：

In [1]:
class LegacyLogger:
    def write_log(self, message):
        print(f"[LEGACY] {message}")

In [2]:
# 但现在你希望使用一个新的日志接口：
class NewLogger:
    def log(self, level, message):
        print(f"[{level.upper()}] {message}")

如果没有适配器，你就需要修改所有调用旧日志系统的代码，这既耗时又容易出错。

适配器的优势

代码复用：可以重用现有的类，无需修改源代码

解耦合：将接口转换逻辑与业务逻辑分离

灵活性：可以轻松切换不同的实现

符合开闭原则：对扩展开放，对修改关闭 

适配器模式的实现方式

类适配器（使用继承）

类适配器通过多重继承来实现接口适配：   

In [1]:
class LegacyLogger:
    '''被适配的旧日志类'''
    def write_log(self, message):
        print(f"[LEGACY] {message}")
       
class NewLogger:
    '''新的日志接口'''
    def log(self, level, message):
        print(f"[{level.upper()}] {message}")

class LoggerAdapter(NewLogger, LegacyLogger):
    '''适配器类，继承自新的日志接口和旧的日志类'''
    def log(self, level, message):
        # 将新接口转换为旧接口
        formatted_message = f"{level}: {message}"
        self.write_log(formatted_message)

# 使用示例
logger = LoggerAdapter()
logger.log("INFO", "系统启动成功")
logger.log("ERROR", "数据库连接失败")

[LEGACY] INFO: 系统启动成功
[LEGACY] ERROR: 数据库连接失败


对象适配器（使用组合）

对象适配器通过组合方式实现，这是更常用的方式：

In [1]:
class LegacyPayment:
    '''被适配的旧支付类'''
    def process_payment(self, amount_in_dollars):
        print(f"处理支付: ${amount_in_dollars}")
        return True
    
class NewPaymentInterface:
    '''新的支付接口'''
    def pay(self, amount, currency="CNY"):
        pass


class PaymentAdapter(NewPaymentInterface):
    """对象适配器 - 通过组合实现"""
    def __init__(self,legacy_payment):
        self.legacy_payment = legacy_payment
    
    def pay(self, amount, currency="CNY"):
        # 货币转换逻辑
        if currency == "CNY":
            amount_in_dollars = amount / 7.0  # 简化汇率转换
        else:
            amount_in_dollars = amount
       
        # 调用旧系统的接口
        return self.legacy_payment.process_payment(amount_in_dollars)
    

# 使用示例
legacy_payment = LegacyPayment()
payment = PaymentAdapter(legacy_payment)

# 使用新接口调用旧系统
payment.pay(700, "CNY")  # 支付700元人民币
payment.pay(100, "USD")  # 支付100美元


处理支付: $100.0
处理支付: $100


True

实际应用案例

案例1：数据格式适配器

In [3]:
import json

class XMLData:
    """XML数据源"""
    def get_xml_data(self):
        return """
        <users>
            <user>
                <name>张三</name>
                <age>25</age>
            </user>
            <user>
                <name>李四</name>
                <age>30</age>
            </user>
        </users>
        """

class JSONProcessor:
    """JSON处理器（期望JSON格式）"""
    def process_json(self, json_data):
        data = json.loads(json_data)
        for user in data['users']:
            print(f"姓名: {user['name']}, 年龄: {user['age']}")

class XMLToJSONAdapter:
    """XML到JSON的适配器"""
   
    def __init__(self, xml_data):
        self.xml_data = xml_data
   
    def convert_to_json(self):
        # 简化的XML到JSON转换
        xml_content = self.xml_data.get_xml_data()
        # 在实际应用中，这里应该使用xml.etree.ElementTree等库进行解析
        json_data = {
            "users": [
                {"name": "张三", "age": 25},
                {"name": "李四", "age": 30}
            ]
        }
        return json.dumps(json_data, ensure_ascii=False)

# 使用适配器
xml_source = XMLData()
adapter = XMLToJSONAdapter(xml_source)
json_processor = JSONProcessor()

json_data = adapter.convert_to_json()
json_processor.process_json(json_data)

姓名: 张三, 年龄: 25
姓名: 李四, 年龄: 30
